In [1]:
# ============================================================
# 1. LIBRERÍAS
# ============================================================
# Si no tienes linearmodels instalado, ejecuta en tu terminal:
# pip install linearmodels statsmodels pandas numpy

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS

np.random.seed(42)


# ============================================================
# 2. GENERAR DATA SINTÉTICA
# ============================================================
n_buckets = 5
n_periodos = 36

# Efectos fijos propios de cada bucket
alpha_i = np.array([0.10, 0.05, 0.00, -0.05, -0.10])

# Efecto común de cada mes
gamma_t = 0.025 * np.sin(np.linspace(0, 4 * np.pi, n_periodos))

# Elasticidades verdaderas que queremos recuperar
elasticidades_verdaderas = {
    "Bucket_0": -6.5,
    "Bucket_1": -4.2,
    "Bucket_2": -2.0,
    "Bucket_3": -1.8,
    "Bucket_4": -0.2,
}

registros = []

for i in range(n_buckets):
    bucket = f"Bucket_{i}"
    beta = elasticidades_verdaderas[bucket]

    for t in range(n_periodos):

        mes = (
            pd.Timestamp("2023-01-01")
            + pd.DateOffset(months=t)
        )

        # Logaritmo de la tasa ofertada
        log_tasa = (
            np.log(0.15)
            + 0.10 * np.sin(t / 4)
            + 0.05 * alpha_i[i]
            + np.random.normal(0, 0.035)
        )

        # Logaritmo de la efectividad
        log_efectividad = (
            np.log(0.45)
            + alpha_i[i]
            + gamma_t[t]
            + beta * (log_tasa - np.log(0.15))
            + np.random.normal(0, 0.025)
        )

        registros.append({
            "bucket": bucket,
            "mes": mes,
            "tasa_ofertada": np.exp(log_tasa),
            "efectividad": np.exp(log_efectividad),
        })

df = pd.DataFrame(registros)

# Transformaciones logarítmicas
df["log_tasa"] = np.log(df["tasa_ofertada"])
df["log_efectividad"] = np.log(df["efectividad"])

print("Dimensiones:", df.shape)
print(df.head())



Dimensiones: (180, 6)
     bucket        mes  tasa_ofertada  efectividad  log_tasa  log_efectividad
0  Bucket_0 2023-01-01       0.153396     0.428499 -1.874735        -0.847467
1  Bucket_0 2023-02-01       0.158071     0.370720 -1.844710        -0.992309
2  Bucket_0 2023-03-01       0.156865     0.375773 -1.852373        -0.978769
3  Bucket_0 2023-04-01       0.170557     0.224890 -1.768684        -1.492143
4  Bucket_0 2023-05-01       0.161314     0.322123 -1.824404        -1.132821


In [2]:

# ============================================================
# 3. PREPARAR VARIABLES DE INTERACCIÓN
# ============================================================
# Una pendiente distinta por bucket.
# Utilizamos nombres seguros, independientemente del nombre
# original de cada bucket.

buckets = sorted(df["bucket"].unique())

nombres_interaccion = {}

for i, bucket in enumerate(buckets):
    nombre = f"tasa_inter_{i}"

    df[nombre] = (
        df["log_tasa"]
        * (df["bucket"] == bucket).astype(int)
    )

    nombres_interaccion[bucket] = nombre


In [3]:

# ============================================================
# 4. MODELO A: OLS CON DUMMIES DE BUCKET Y MES
# ============================================================
# C(bucket): efectos fijos de bucket
# C(mes):    efectos fijos de mes
# tasa_inter_i: elasticidad específica de cada bucket

formula = (
    "log_efectividad ~ C(bucket) + C(mes) + "
    + " + ".join(nombres_interaccion.values())
)

modelo_ols = smf.ols(
    formula=formula,
    data=df
).fit()

print("\n" + "=" * 70)
print("MODELO A: OLS CON DUMMIES")
print("=" * 70)
print(modelo_ols.summary())




MODELO A: OLS CON DUMMIES
                            OLS Regression Results                            
Dep. Variable:        log_efectividad   R-squared:                       0.995
Model:                            OLS   Adj. R-squared:                  0.993
Method:                 Least Squares   F-statistic:                     591.5
Date:                Sat, 24 Oct 2026   Prob (F-statistic):          4.12e-135
Time:                        23:13:22   Log-Likelihood:                 447.63
No. Observations:                 180   AIC:                            -805.3
Df Residuals:                     135   BIC:                            -661.6
Df Model:                          44                                         
Covariance Type:            nonrobust                                         
                                                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------

In [4]:

# ============================================================
# 5. MODELO B: PANEL OLS CON EFECTOS FIJOS ABSORBIDOS
# ============================================================
# La entidad es el bucket y el tiempo es el mes.
# PanelOLS absorbe los efectos fijos de ambas dimensiones.

df_panel = (
    df.set_index(["bucket", "mes"])
      .sort_index()
)

variables_exog = list(nombres_interaccion.values())

modelo_panel = PanelOLS(
    dependent=df_panel["log_efectividad"],
    exog=df_panel[variables_exog],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True,
)

resultado_panel = modelo_panel.fit(
    cov_type="unadjusted"
)

print("\n" + "=" * 70)
print("MODELO B: PANEL OLS CON EFECTOS FIJOS")
print("=" * 70)
print(resultado_panel.summary)




MODELO B: PANEL OLS CON EFECTOS FIJOS
                          PanelOLS Estimation Summary                           
Dep. Variable:        log_efectividad   R-squared:                        0.9869
Estimator:                   PanelOLS   R-squared (Between):             -74.072
No. Observations:                 180   R-squared (Within):               0.9895
Date:                Sat, Oct 24 2026   R-squared (Overall):             -66.930
Time:                        23:13:23   Log-likelihood                    447.63
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      2034.4
Entities:                           5   P-value                           0.0000
Avg Obs:                       36.000   Distribution:                   F(5,135)
Min Obs:                       36.000                                           
Max Obs:                       36.000   F-statistic (robust):         

In [5]:

# ============================================================
# 6. COMPARAR LAS ELASTICIDADES ESTIMADAS
# ============================================================

filas = []

for bucket, nombre in nombres_interaccion.items():

    beta_real = elasticidades_verdaderas[bucket]

    # Modelo OLS con dummies
    beta_ols = modelo_ols.params.get(nombre, np.nan)
    pvalue_ols = modelo_ols.pvalues.get(nombre, np.nan)

    # Modelo PanelOLS
    beta_panel = resultado_panel.params.get(nombre, np.nan)
    pvalue_panel = resultado_panel.pvalues.get(nombre, np.nan)

    filas.append({
        "bucket": bucket,
        "elasticidad_verdadera": beta_real,
        "elasticidad_OLS": beta_ols,
        "pvalue_OLS": pvalue_ols,
        "elasticidad_PanelOLS": beta_panel,
        "pvalue_PanelOLS": pvalue_panel,
        "diferencia_coeficientes": beta_ols - beta_panel,
    })

comparacion = pd.DataFrame(filas)

print("\n" + "=" * 70)
print("COMPARACIÓN DE ELASTICIDADES")
print("=" * 70)

print(
    comparacion.round(5).to_string(index=False)
)



COMPARACIÓN DE ELASTICIDADES
  bucket  elasticidad_verdadera  elasticidad_OLS  pvalue_OLS  elasticidad_PanelOLS  pvalue_PanelOLS  diferencia_coeficientes
Bucket_0                   -6.5         -6.37152     0.00000              -6.37152          0.00000                      0.0
Bucket_1                   -4.2         -4.15581     0.00000              -4.15581          0.00000                      0.0
Bucket_2                   -2.0         -1.87018     0.00000              -1.87018          0.00000                      0.0
Bucket_3                   -1.8         -1.77729     0.00000              -1.77729          0.00000                      0.0
Bucket_4                   -0.2         -0.14624     0.07768              -0.14624          0.07768                      0.0


In [6]:

# ============================================================
# 7. VALIDACIÓN: ¿QUÉ TAN PARECIDOS SON LOS COEFICIENTES?
# ============================================================

comparacion["error_abs_OLS"] = (
    comparacion["elasticidad_OLS"]
    - comparacion["elasticidad_verdadera"]
).abs()

comparacion["error_abs_PanelOLS"] = (
    comparacion["elasticidad_PanelOLS"]
    - comparacion["elasticidad_verdadera"]
).abs()

print("\nError absoluto promedio frente a las elasticidades reales:")
print(
    "OLS con dummies:",
    comparacion["error_abs_OLS"].mean().round(5)
)
print(
    "PanelOLS:",
    comparacion["error_abs_PanelOLS"].mean().round(5)
)

print("\nMáxima diferencia entre los coeficientes de ambos modelos:")
print(
    comparacion["diferencia_coeficientes"].abs().max().round(8)
)


Error absoluto promedio frente a las elasticidades reales:
OLS con dummies: 0.07579
PanelOLS: 0.07579

Máxima diferencia entre los coeficientes de ambos modelos:
0.0
